**Text Preprocessing**

**Goal:** Clean raw review text so it's ready for TF-IDF and model training.

**Lowercase → Remove URLs → Remove emojis → Remove punctuation → Remove numbers →Remove stopwords → Tokenize**

In [6]:
# Loading all the libraries
import pandas as pd
import numpy as np
import re
import nltk
from nltk.tokenize import word_tokenize
import os
os.chdir('..') if os.path.basename(os.getcwd()) == 'notebooks' else None
print(os.getcwd())

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("All imports successful")

C:\Users\user\Desktop\nepali-sentiment-analysis
All imports successful


In [7]:
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

english_stopwords = set(stopwords.words('english'))

nepali_roman_stopwords = {
    'cha', 'chha', 'thiyo', 'ho', 'hola', 'bhayo', 'garcha', 'gareko',
    'huncha', 'huna', 'hunna', 'bhayena', 'rahecha', 'raicha', 'raheko',
    'ma', 'ko', 'ka', 'ki', 'lai', 'bata', 'sanga', 'le', 'pani',
    'ra', 'ani', 'tara', 'ta', 'ni', 'nai', 'chai', 'mero', 'timro',
    'usko', 'hamro', 'yo', 'tyo', 'uni', 'hami', 'tapai', 'tapain',
    'hai', 'la', 'na', 'ne', 'nah', 'yaar', 'aba', 'ali', 'pachi',
}

all_stopwords = english_stopwords.union(nepali_roman_stopwords)
print(f'Total stopwords: {len(all_stopwords)}')

Total stopwords: 247


In [8]:
train_df = pd.read_csv('data/raw/kathmandu_reviews_dataset/split_train.csv')
val_df   = pd.read_csv('data/raw/kathmandu_reviews_dataset/split_val.csv')
test_df  = pd.read_csv('data/raw/kathmandu_reviews_dataset/split_test.csv')

print(f'Train : {train_df.shape}')
print(f'Val   : {val_df.shape}')
print(f'Test  : {test_df.shape}')

Train : (70000, 22)
Val   : (15000, 22)
Test  : (15000, 22)


In [9]:
print("RAW SAMPLES")
for label in ['positive', 'negative', 'neutral']:
    sample = train_df[train_df['sentiment_label'] == label]['review_text'].iloc[0]
    print(f'\n[{label.upper()}]\n{sample}')

RAW SAMPLES

[POSITIVE]
Best purchase of the year ho yo! Door step delivery raamro thiyo, no hassle. Refund/exchange policy customer-friendly cha. Sabai lai recommend gareko chu yo seller bata kinna. Screen broken cha, paisa khera gayo.

[NEGATIVE]
Battery dead aayo, charge nai hudaina. Delivery time really duplicate jasto cha. Worst experience, refund maag rakheko chu. Fast delivery ra original product, dubai bhayo.

[NEUTRAL]
Daily use ko lagi okay cha jastai laagcha. Warranty card ayo but verification needed jasto. Use garera bhanaula time pachi.


In [10]:
#Lowercase
def lowercase_text(text):
    if pd.isna(text):
        return ''
    return str(text).lower()

# Test
sample = "Best purchase of the Year! EKDAM Ramro cha."
print(lowercase_text(sample))

best purchase of the year! ekdam ramro cha.


In [11]:
#Remove emojis
def remove_emojis(text):
    return re.sub(r'[^\x00-\x7F]+', ' ', text)

# Test
sample = "Ekdam ramro cha 👍 delivery fast thiyo 🔥"
print(remove_emojis(sample))

Ekdam ramro cha   delivery fast thiyo  


In [12]:
#Remove URLs
def remove_urls(text):
    return re.sub(r'http\S+|www\.\S+', '', text)

# Test
sample = "Ramro product cha, check here: https://daraz.com.np/product123"
print(remove_urls(sample))

Ramro product cha, check here: 


In [13]:
#Remove Punctuations
def remove_punctuation(text):
    return re.sub(r'[^\w\s]', ' ', text)

# Test 
sample = "best purchase of the year! ekdam ramro cha, delivery fast thiyo."
print(remove_punctuation(sample))

best purchase of the year  ekdam ramro cha  delivery fast thiyo 


In [14]:
#Remove Numbers
def remove_numbers(text):
    return re.sub(r'\b\d+\b', '', text)

# Test 
sample = "delivery 5 din late thiyo, 2 palta call garyo"
print(remove_numbers(sample))

delivery  din late thiyo,  palta call garyo


In [15]:
#Remove extra whitespace
def remove_extra_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

# Test
sample = "delivery  din late thiyo,   palta  call garyo"
print(remove_extra_whitespace(sample))

delivery din late thiyo, palta call garyo


In [16]:
def remove_stopwords(text):
    tokens = text.split()
    filtered = [word for word in tokens if word not in all_stopwords]
    return ' '.join(filtered)

# Test it
sample = "best purchase of the year ho yo door step delivery raamro thiyo"
print(remove_stopwords(sample))

best purchase year door step delivery raamro


In [17]:
#Combine all functions 
def preprocess_text(text):
    text = lowercase_text(text)
    text = remove_emojis(text)
    text = remove_urls(text)
    text = remove_punctuation(text)
    text = remove_numbers(text)
    text = remove_extra_whitespace(text)
    text = remove_stopwords(text)
    return text

# Test on all 3 sample types
for label in ['positive', 'negative', 'neutral']:
    raw = train_df[train_df['sentiment_label'] == label]['review_text'].iloc[0]
    cleaned = preprocess_text(raw)
    print(f'\n[{label.upper()}]')
    print(f'BEFORE: {raw}')
    print(f'AFTER : {cleaned}')


[POSITIVE]
BEFORE: Best purchase of the year ho yo! Door step delivery raamro thiyo, no hassle. Refund/exchange policy customer-friendly cha. Sabai lai recommend gareko chu yo seller bata kinna. Screen broken cha, paisa khera gayo.
AFTER : best purchase year door step delivery raamro hassle refund exchange policy customer friendly sabai recommend chu seller kinna screen broken paisa khera gayo

[NEGATIVE]
BEFORE: Battery dead aayo, charge nai hudaina. Delivery time really duplicate jasto cha. Worst experience, refund maag rakheko chu. Fast delivery ra original product, dubai bhayo.
AFTER : battery dead aayo charge hudaina delivery time really duplicate jasto worst experience refund maag rakheko chu fast delivery original product dubai

[NEUTRAL]
BEFORE: Daily use ko lagi okay cha jastai laagcha. Warranty card ayo but verification needed jasto. Use garera bhanaula time pachi.
AFTER : daily use lagi okay jastai laagcha warranty card ayo verification needed jasto use garera bhanaula time

In [20]:
#Apply to all
train_df['cleaned_text'] = train_df['review_text'].apply(preprocess_text)
val_df['cleaned_text']   = val_df['review_text'].apply(preprocess_text)
test_df['cleaned_text']  = test_df['review_text'].apply(preprocess_text)

print(f'Train cleaned: {train_df.shape}')
print(f'Val   cleaned: {val_df.shape}')
print(f'Test  cleaned: {test_df.shape}')

Train cleaned: (70000, 23)
Val   cleaned: (15000, 23)
Test  cleaned: (15000, 23)


In [21]:
#Save the cleaned data
train_df.to_csv('data/processed/train_cleaned.csv', index=False)
val_df.to_csv('data/processed/val_cleaned.csv', index=False)
test_df.to_csv('data/processed/test_cleaned.csv', index=False)

print('Saved:')
print('data/processed/train_cleaned.csv')
print('data/processed/val_cleaned.csv')
print('data/processed/test_cleaned.csv')

Saved:
data/processed/train_cleaned.csv
data/processed/val_cleaned.csv
data/processed/test_cleaned.csv


In [23]:
preprocess_code = '''import re
import nltk
import pandas as pd
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)

english_stopwords = set(stopwords.words("english"))

nepali_roman_stopwords = {
    "cha", "chha", "thiyo", "ho", "hola", "bhayo", "garcha", "gareko",
    "huncha", "huna", "hunna", "bhayena", "rahecha", "raicha", "raheko",
    "ma", "ko", "ka", "ki", "lai", "bata", "sanga", "le", "pani",
    "ra", "ani", "tara", "ta", "ni", "nai", "chai", "mero", "timro",
    "usko", "hamro", "yo", "tyo", "uni", "hami", "tapai", "tapain",
    "hai", "la", "na", "ne", "nah", "yaar", "aba", "ali", "pachi",
}

all_stopwords = english_stopwords.union(nepali_roman_stopwords)

def lowercase_text(text):
    if pd.isna(text):
        return ""
    return str(text).lower()

def remove_emojis(text):
    return re.sub(r"[^\\x00-\\x7F]+", " ", text)

def remove_urls(text):
    return re.sub(r"http\\S+|www\\.\\S+", "", text)

def remove_punctuation(text):
    return re.sub(r"[^\\w\\s]", " ", text)

def remove_numbers(text):
    return re.sub(r"\\b\\d+\\b", "", text)

def remove_extra_whitespace(text):
    return re.sub(r"\\s+", " ", text).strip()

def remove_stopwords(text):
    tokens = text.split()
    filtered = [word for word in tokens if word not in all_stopwords]
    return " ".join(filtered)

def preprocess_text(text):
    text = lowercase_text(text)
    text = remove_emojis(text)
    text = remove_urls(text)
    text = remove_punctuation(text)
    text = remove_numbers(text)
    text = remove_extra_whitespace(text)
    text = remove_stopwords(text)
    return text
'''

with open('src/preprocess.py', 'w') as f:
    f.write(preprocess_code)

print('src/preprocess.py updated!')

src/preprocess.py updated!
